# Borrowed Authority · real Cohere Tiny Aya experiment

**Lexsi AI Safety Track B.** This notebook executes a real Cohere open-weight model. It does **not** simulate model responses. Requires a GPU and permission to access [CohereLabs/tiny-aya-earth](https://huggingface.co/CohereLabs/tiny-aya-earth).

**The current project has no measured AI results.** Running these cells creates the first real results. Do not use the public demo's reference labels as model predictions.

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Change Runtime → GPU in Colab before continuing.')

### Accept the model license and log in privately

Visit https://huggingface.co/CohereLabs/tiny-aya-earth and accept the gated access agreement in your Hugging Face account. Never post a token in chat, GitHub or the public notebook.

In [ ]:
!pip -q install -U transformers accelerate huggingface_hub curatorkit auditkit
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!git clone --branch lexsi-borrowed-authority --depth 1 https://github.com/SheunDiran/Projects.git borrowed-authority
%cd borrowed-authority
!python scripts/generate_dataset.py
!python scripts/build_advanced_dataset.py
!python scripts/prepare_training_data.py
!python -m unittest discover -s tests -v

### Toolkit A — CuratorKIT

This is a **real CPU dataset curation run**, not model inference or fine-tuning. Check rejected rows and provenance manifest before using the output for training.

In [ ]:
!python toolkits/run_curatorkit.py

### Baseline real Cohere inference

Runs on held-out 46 cases. The first call downloads gated model weights; allow appropriate runtime. The two conditions use identical weights but different instructions. They are **prompt interventions, not fine-tuned checkpoints**.

In [ ]:
!python scripts/run_tiny_aya.py --cases data/benchmark_test.jsonl --regime baseline --output results/tiny_aya_baseline.jsonl

### Provenance-hardened condition

In [ ]:
!python scripts/run_tiny_aya.py --cases data/benchmark_test.jsonl --regime hardened --output results/tiny_aya_hardened.jsonl

### Toolkit B — AuditKIT + paired metrics

Metrics below are **only** valid if both inference runs above completed. Do not copy numbers from scripted examples.

In [ ]:
!python scripts/evaluate.py --cases data/benchmark_test.jsonl --predictions results/tiny_aya_baseline.jsonl --after results/tiny_aya_hardened.jsonl --output results/paired_metrics.json
!python toolkits/run_auditkit.py --cases data/benchmark_test.jsonl --predictions results/tiny_aya_baseline.jsonl --output results/auditkit_baseline.json
!python toolkits/run_auditkit.py --cases data/benchmark_test.jsonl --predictions results/tiny_aya_hardened.jsonl --output results/auditkit_hardened.json

### Prepare verifiable Hugging Face artifacts

This only packages the files. Publishing needs your own account and explicit approval. Upload the dataset and run manifests with matching revision/versions; do not pretend a model adapter exists unless you trained one.

In [ ]:
!python scripts/prepare_hf_artifacts.py
!ls -lh artifacts/hf_dataset_upload

### Next scientific steps (not automated)

1. Human-review labels, add naturally written cases; 2. report family-level generalization and regressions; 3. if compute is sufficient test actual SafeTune intervention and independent held-out verification; 4. publish results to Hugging Face and finalize the paper. The current notebook does not claim a fine-tuned model.